# OpenPlaque — Left Coronary Ostium Mask Consensus
Fresh-baseline, single-kernel experiment. Discovers a second coronary-sized aortic-root exit from coronary-mask/aortic-surface interface clusters. RCA is the positive control. LAD/C6 are post-hoc QC only. No Python subprocesses.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/Left_Coronary_Ostium_Mask_Consensus_v1'
BRANCH = 'left-coronary-ostium-mask-consensus-from-main'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM = 'left-coronary-ostium-mask-consensus-v1.0-lowmem'
print('Branch:', BRANCH)
print('Output:', OUTPUT_DIR)


In [ ]:
import os, shutil
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone --depth 1 --branch $BRANCH https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
HEAD = !git -C /content/OpenPlaque rev-parse HEAD
HEAD = HEAD[0].strip()
print('Checked out HEAD:', HEAD)


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy


In [ ]:
import sys, importlib, pytest
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_coronary_ostium_mask_consensus as exp
print('openplaque:', openplaque.__file__)
print('algorithm:', exp.ALGORITHM)
assert exp.BASELINE == BASELINE
assert exp.ALGORITHM == EXPECTED_ALGORITHM
print('self-test:', exp.synthetic_interface_cluster_self_test())
rc = pytest.main(['-q','/content/OpenPlaque/tests/test_left_coronary_ostium_mask_consensus.py'])
if rc != 0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'Joint_Three_Vessel_Template_Classifier_v1/candidate_04_source_path.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz',
]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))


In [ ]:
import gc
from openplaque.left_coronary_ostium_mask_consensus import run
gc.collect()
result = run(DRIVE_ROOT, OUTPUT_DIR)
print('STATUS:', result['summary']['status'])
print('RCA CONTROL:', result['summary'].get('RCA_interface_control',{}).get('control_pass'))
print('ROOT INTERFACE CLUSTERS:', result['summary'].get('n_root_interface_clusters'))
print('LEFT CANDIDATE:', result['summary'].get('left_candidate'))
print('REPORT:', result.get('report'))
print('ZIP:', result.get('zip'))
